# Multi-head attention, step by step

Keep the river-bank example from Part II. Follow two complete heads through the
same figures as Part III, then implement the calculation and compare it with
PyTorch. The toy uses hand-chosen parameters. The final links lead to genuinely
trained TinyStories models, not this worksheet.

[Part III](../../part3.html) · [Live models](../../word-lab/) · [Download code and notebooks](wordlm-notebooks.zip)

Run all cells from this directory. No data download or training run is needed.
The single optimizer step demonstrates learning; it does not create the models
used in the benchmark.

In [1]:
import json
import math
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display, HTML
from multihead_from_scratch import (TinyMultiHeadLM, ScratchMultiHead,
                                   load_worksheet_weights, copy_to_pytorch)

torch.manual_seed(7)
worksheet = json.loads(Path('multihead-worksheet.json').read_text())
word_to_id = {word: i for i, word in enumerate(worksheet['vocab'])}
river_ids = [word_to_id[w.lower()] for w in worksheet['sentences']['river']]
cheque_ids = [word_to_id[w.lower()] for w in worksheet['sentences']['cheque']]
print('River tokens:', worksheet['sentences']['river'])
print('River IDs:', river_ids)

River tokens: ['The', 'fisherman', 'sat', 'beside', 'the', 'river', 'bank', 'and', 'watched', 'the']
River IDs: [0, 1, 2, 3, 0, 4, 5, 6, 7, 0]


<a id="s01-name"></a>
## Which earlier characters might help?

[Matching slide](../../part3.html?present#s01/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="a  b  i → Embeddings → Read context → Next character" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>a  b  i → Embeddings → Read context → Next character</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">a  b  i</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3 known IDs</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Embeddings</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3 learned rows</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Read context</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">MLP or attention</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Next character</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">for example, d</text></g></svg>

Part I predicted the next character in a name such as aabid. The question stays the same; we are changing how the model reads the known context.

<a id="s01-river"></a>
## Back to the river bank

[Matching slide](../../part3.html?present#s01/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 370" role="img" aria-label="The river sentence from Part II" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>The river sentence from Part II</title><g><rect x="20" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="60.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">The</text><text x="60.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">1</text></g><g><rect x="112" y="55" width="150" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="187.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">fisherman</text><text x="187.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2</text></g><g><rect x="274" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="314.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">sat</text><text x="314.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3</text></g><g><rect x="366" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="421.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">beside</text><text x="421.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4</text></g><g><rect x="488" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="528.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">the</text><text x="528.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">5</text></g><g><rect x="580" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="635.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">river</text><text x="635.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">6</text></g><g><rect x="702" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="757.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">bank</text><text x="757.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">7</text></g><g><rect x="824" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="864.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">and</text><text x="864.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">8</text></g><g><rect x="916" y="55" width="135" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="983.5" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">watched</text><text x="983.5" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">9</text></g><g><rect x="1063" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="1103.0" y="87" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">the</text><text x="1103.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">10</text></g><text x="580" y="325" fill="var(--ink)" font-size="30" text-anchor="middle" font-weight="500">Predict the next token after the final “the”.</text><text x="635.0" y="245" fill="var(--c-q)" font-size="30" text-anchor="middle" font-weight="500">Which setting?</text><text x="187.0" y="245" fill="var(--c-q)" font-size="30" text-anchor="middle" font-weight="500">Who is in the scene?</text><path d="M635.0 220 L635.0 145 M631.1058165769135 154.21060994002886 L635.0 145 L638.8941834230865 154.21060994002886" fill="none" stroke="var(--c-q)" stroke-width="2.5"/><path d="M187.0 220 L187.0 145 M183.1058165769135 154.21060994002886 L187.0 145 L190.8941834230865 154.21060994002886" fill="none" stroke="var(--c-q)" stroke-width="2.5"/></svg>

One query can benefit from several clues at once. What setting are we in? Who is there?

<a id="s01-scope"></a>
## Keep the two examples separate

[Matching slide](../../part3.html?present#s01/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Worksheet; Trained experiment" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Worksheet; Trained experiment</title><g data-cell-left="20" data-cell-right="580"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Worksheet</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Trained experiment</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2 heads × 2 coordinates</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4 heads × 16 coordinates</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Hand-chosen projections</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Learned from TinyStories</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Part II token + position rows</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Same 64-token benchmark window</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The worksheet explains the arithmetic. Held-out results later tell us whether the trained models improved.

<a id="s02-break"></a>
## From one weight row to two

[Matching slide](../../part3.html?present#s02/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

What changes when the same input passes through two sets of projections?

<a id="s02-one"></a>
## One head produces one message

[Matching slide](../../part3.html?present#s02/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Full E → Q, K, V → Weights A → Message AV" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Full E → Q, K, V → Weights A → Message AV</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Full E</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">all coordinates</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q, K, V</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">learned projections</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Weights A</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one row per query</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Message AV</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one weighted sum</text></g></svg>

One head can already read several sources. Its value coordinates share the same attention weights.

<a id="s02-two"></a>
## Two heads keep two messages

[Matching slide](../../part3.html?present#s02/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₁ · K₁ · V₁</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₂ · K₂ · V₂</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

Each head has its own Q, K and V projections and its own softmax. Both receive the full E.

<a id="s02-full"></a>
## Project first; split the projected coordinates

[Matching slide](../../part3.html?present#s02/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E [B,T,4] → W_Q [4,4] → Q [B,T,4] → 2 heads × 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E [B,T,4] → W_Q [4,4] → Q [B,T,4] → 2 heads × 2</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [B,T,4]</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">full input rows</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q [4,4]</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">learned mixing</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q [B,T,4]</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">projected coordinates</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">2 heads × 2</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one view per head</text></g></svg>

We split Q, K and V after projection. We do not give head 1 the first half of the raw embedding and head 2 the second half.

<a id="s02-columns"></a>
## Separate heads can use different input features

[Matching slide](../../part3.html?present#s02/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Input coordinate; W_Q: head 1; W_Q: head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input coordinate; W_Q: head 1; W_Q: head 2</title><g data-cell-left="20" data-cell-right="370"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Input coordinate</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">W_Q: head 1</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">W_Q: head 2</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">finance</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">person</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">glue</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 1]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 0]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

These are two 4×2 matrices placed side by side. Every head can learn from every input coordinate.

<a id="s03-break"></a>
## Two heads, one receiving token

[Matching slide](../../part3.html?present#s03/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Keep the final “the” at position 10 as the query throughout.

<a id="s03-input"></a>
## Start with the same position-aware rows

[Matching slide](../../part3.html?present#s03/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Position / token; water; finance; person; glue" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Position / token; water; finance; person; glue</title><g data-cell-left="20" data-cell-right="350"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Position / token</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">water</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">finance</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">person</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">glue</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2 fisherman</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.0</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.1</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.1</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">6 river</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.1</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">−0.1</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.1</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">10 the</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

E already includes token embedding + position embedding. Token IDs are not embedding coordinates.

<a id="s03-q1"></a>
## The first query asks about the setting

[Matching slide](../../part3.html?present#s03/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e₁₀ → W_Q¹ → q₁₀¹" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e₁₀ → W_Q¹ → q₁₀¹</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e₁₀</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q¹</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 × 2</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">q₁₀¹</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2.300, 2.300]</text></g></svg>

Our chosen projection copies the glue coordinate into both query coordinates: [2.3, 2.3].

<a id="s03-q2"></a>
## The second query uses a different projection

[Matching slide](../../part3.html?present#s03/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e₁₀ → W_Q² → q₁₀²" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e₁₀ → W_Q² → q₁₀²</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e₁₀</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q²</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 × 2</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">q₁₀²</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2.300, 0.000]</text></g></svg>

The same input row now produces [2.3, 0.0]. The query changes because the projection matrix changes.

<a id="s03-keys"></a>
## The source keys are different too

[Matching slide](../../part3.html?present#s03/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Source; Key in head 1; Key in head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Key in head 1; Key in head 2</title><g data-cell-left="20" data-cell-right="360"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Key in head 1</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Key in head 2</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">fisherman</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.000, 0.100]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.100, 0.000]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.100, −0.100]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.100]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">bank</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.700, 0.700]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.100, 0.800]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">the</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.000]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 2.300]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

Head 1 exposes water/finance features. Head 2 exposes person/glue features. These labels belong to this designed worksheet.

<a id="s03-dot1"></a>
## Head 1 scores the river key

[Matching slide](../../part3.html?present#s03/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Calculation; Value" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Calculation; Value</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Calculation</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">q₁₀¹</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.3, 2.3]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">k₆¹ for river</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.1, −0.1]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Dot product</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3 × 3.1 + 2.3 × (−0.1) = 6.9</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Divide by √2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4.879</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

The scaling uses the head width: two coordinates, not four.

<a id="s03-dot2"></a>
## Head 2 scores the fisherman key

[Matching slide](../../part3.html?present#s03/7/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Calculation; Value" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Calculation; Value</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Calculation</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">q₁₀²</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.3, 0.0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">k₂² for fisherman</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.1, 0.0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Dot product</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3 × 2.1 + 0.0 × 0.0 = 4.83</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Divide by √2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.415</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

We compare each head’s query only with keys from that same head.

<a id="s03-weights1"></a>
## Head 1: scores become a weight row

[Matching slide](../../part3.html?present#s03/8/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.49</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.81</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.33</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">4.88</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">2.28</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

Each weight is exp(score) divided by the sum of exp(scores) in this row. All ten sources are allowed for the final query.

<a id="s03-weights2"></a>
## Head 2 has its own softmax

[Matching slide](../../part3.html?present#s03/9/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 2</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 2 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.4957763940137875"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.693</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.98</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11625688180810867"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.060</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.12266008757658185"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.071</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

Normalize over sources within head 2. Do not normalize across heads or average their scores.

<a id="s03-value1"></a>
## Head 1 sends water and finance information

[Matching slide](../../part3.html?present#s03/10/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Source; Weight; Value row; Weighted contribution" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Weight; Value row; Weighted contribution</title><g data-cell-left="20" data-cell-right="255"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weight</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value row</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weighted contribution</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="101" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500">0.718</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[3.100, −0.100]</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.225, −0.072]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">All other sources</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="152" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[0.392, 0.054]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Total message</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="203" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The river contribution is its weight × its value row. Add the contributions from every source to get the two-coordinate message.

<a id="s03-value2"></a>
## Head 2 sends a different kind of message

[Matching slide](../../part3.html?present#s03/11/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Source; Weight; Value row; Weighted contribution" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Weight; Value row; Weighted contribution</title><g data-cell-left="20" data-cell-right="255"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weight</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value row</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weighted contribution</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">fisherman</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="101" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500">0.693</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.100, 0.000]</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[1.455, 0.000]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">All other sources</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="152" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[0.097, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Total message</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="203" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[1.552, 0.538]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The second head mixes its own values with its own weights. Matching features choose where to read; values carry the information.

<a id="s04-break"></a>
## Bring the two messages back together

[Matching slide](../../part3.html?present#s04/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

How do two small message rows become one update to the original row?

<a id="s04-join"></a>
## Concatenate the messages, not the tokens

[Matching slide](../../part3.html?present#s04/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Head; Message coordinates; Message" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Head; Message coordinates; Message</title><g data-cell-left="20" data-cell-right="180"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Head</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Message coordinates</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Message</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water / finance</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">person / glue</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1.552, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Joined</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">head 1, then head 2</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018, 1.552, 0.538]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Two 2-coordinate rows become one 4-coordinate row. Concatenation preserves both feature groups; it does not average them.

<a id="s04-wo"></a>
## W_O can mix information across heads

[Matching slide](../../part3.html?present#s04/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Joined coordinate; Output weights" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Joined coordinate; Output weights</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Joined coordinate</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Output weights</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 0, 0, 0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 1, 0, 0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.25, 0, 1, 0]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0, 0, 1]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

Water update = 2.617 + 0.25 × 1.552 = 3.005. W_O maps the joined message back to model width.

<a id="s04-residual"></a>
## Add the update to the original row

[Matching slide](../../part3.html?present#s04/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Row; Four coordinates" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Row; Four coordinates</title><g data-cell-left="20" data-cell-right="470"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Row</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Four coordinates</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Original e₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Attention update Δe₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.005, −0.018, 1.552, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Updated e′₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.005, −0.018, 1.552, 2.838]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The residual is the same addition as in Part II. There is still one updated row per input token.

<a id="s04-predict"></a>
## The prediction MLP stays in place

[Matching slide](../../part3.html?present#s04/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e′₁₀ → Hidden + ReLU → 20 logits → Softmax" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e′₁₀ → Hidden + ReLU → 20 logits → Softmax</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e′₁₀</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 coordinates</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Hidden + ReLU</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">8 activations</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">20 logits</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one per vocabulary item</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Softmax</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token probabilities</text></g></svg>

Choose a token only after vocabulary softmax. Attention weights choose source positions; vocabulary probabilities choose possible next tokens.

<a id="s04-live"></a>
## Change the context; inspect both heads

[Matching slide](../../part3.html?present#s04/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.49</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.81</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.33</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">4.88</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">2.28</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

The matching slide lets you switch contexts and inspect either head. Here, compute both sentences and print their final-query messages. These are hand-chosen worksheet parameters, not trained results.

In [2]:
for name in ['river', 'cheque']:
    ids = torch.tensor([[word_to_id[w.lower()] for w in worksheet['sentences'][name]]])
    demo = load_worksheet_weights(TinyMultiHeadLM(len(word_to_id)), worksheet)
    E_demo = demo.token_embedding(ids) + demo.position_embedding(torch.arange(10))
    with torch.no_grad():
        _, weights = demo.attention(E_demo)
        values = demo.attention.split_heads(demo.attention.W_V(E_demo))
        messages = weights @ values
    print(name, 'final messages by head:', messages[0, :, -1].tolist())

river final messages by head: [[2.6168477535247803, -0.017858127132058144], [1.5519635677337646, 0.5383409261703491]]
cheque final messages by head: [[0.05300545319914818, 2.471384286880493], [2.3669915199279785, 0.4609062373638153]]


<a id="s05-break"></a>
## From the drawing to tensors

[Matching slide](../../part3.html?present#s05/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Keep B = 2 examples, T = 10 tokens, D = 4 coordinates and H = 2 heads.

<a id="s05-model"></a>
## Create the tables, projections and prediction MLP

[Matching slide](../../part3.html?present#s05/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token + position → ScratchMultiHead → Hidden → vocabulary" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token + position → ScratchMultiHead → Hidden → vocabulary</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">20 × 4 and 10 × 4</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">ScratchMultiHead</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 coordinates, 2 heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Hidden → vocabulary</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 → 8 → 20</text></g></svg>

The download defines every layer. Load the hand-chosen worksheet weights to reproduce the printed numbers; ordinary training starts from random parameters.

### The complete implementation

This is the same source file imported above. Read the small snippets that follow alongside the diagram, then return here to see how they fit together. `load_worksheet_weights` copies the printed parameters so our outputs match the figures. It is not part of an ordinary training loop.

In [3]:
"""Part III: explicit multi-head self-attention and a small next-token model."""
import math
import torch
from torch import nn
from torch.nn import functional as F


class ScratchMultiHead(nn.Module):
    def __init__(self, width=4, heads=2):
        super().__init__()
        if heads < 1 or width % heads:
            raise ValueError('width must be divisible by a positive head count')
        self.width, self.heads = width, heads
        self.head_width = width // heads
        self.W_Q = nn.Linear(width, width, bias=False)
        self.W_K = nn.Linear(width, width, bias=False)
        self.W_V = nn.Linear(width, width, bias=False)
        self.W_O = nn.Linear(width, width, bias=False)

    def split_heads(self, rows):
        B, T, _ = rows.shape
        return rows.reshape(B, T, self.heads, self.head_width).transpose(1, 2)

    def forward(self, E, padding_mask=None):
        B, T, _ = E.shape
        Q = self.split_heads(self.W_Q(E))
        K = self.split_heads(self.W_K(E))
        V = self.split_heads(self.W_V(E))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_width)
        blocked = torch.ones(T, T, dtype=torch.bool, device=E.device).triu(1)
        if padding_mask is not None:
            # Ignore PAD sources for real queries. PAD-query outputs are unused;
            # leave their causal prefix available to avoid an all-masked softmax.
            blocked = blocked | (padding_mask[:, None, :] & ~padding_mask[:, :, None])
            blocked = blocked[:, None]
        weights = scores.masked_fill(blocked, float('-inf')).softmax(dim=-1)
        messages = weights @ V
        joined = messages.transpose(1, 2).contiguous().reshape(B, T, self.width)
        delta = self.W_O(joined)
        return delta, weights


class TinyMultiHeadLM(nn.Module):
    def __init__(self, vocab_size, context=10, width=4, heads=2, hidden=8):
        super().__init__()
        self.context = context
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context, width)
        self.attention = ScratchMultiHead(width, heads)
        self.hidden = nn.Linear(width, hidden)
        self.readout = nn.Linear(hidden, vocab_size)

    def forward(self, ids):
        T = ids.shape[1]
        if T > self.context:
            raise ValueError('Crop the prompt to the configured context window')
        E = self.token_embedding(ids) + self.position_embedding(torch.arange(T, device=ids.device))
        delta, _ = self.attention(E)
        updated = E + delta
        return self.readout(F.relu(self.hidden(updated[:, -1])))


def copy_to_pytorch(scratch):
    """Use exactly the same parameters, not a newly randomized comparison."""
    layer = nn.MultiheadAttention(scratch.width, scratch.heads, bias=False,
                                  dropout=0.0, batch_first=True)
    layer = layer.to(device=scratch.W_Q.weight.device, dtype=scratch.W_Q.weight.dtype)
    with torch.no_grad():
        layer.in_proj_weight.copy_(torch.cat([scratch.W_Q.weight,
                                             scratch.W_K.weight,
                                             scratch.W_V.weight], dim=0))
        layer.out_proj.weight.copy_(scratch.W_O.weight)
    return layer


def load_worksheet_weights(model, worksheet):
    """Load the printed, hand-chosen example; this is not a training algorithm."""
    heads = worksheet['headsLesson']['projections']
    with torch.no_grad():
        model.token_embedding.weight.copy_(torch.tensor([worksheet['tok_emb'][w] for w in worksheet['vocab']]))
        model.position_embedding.weight.copy_(torch.tensor(worksheet['pos_emb'][:model.context]))
        for letter in ['Q', 'K', 'V']:
            packed = [a + b for a, b in zip(heads[0][letter], heads[1][letter])]
            getattr(model.attention, 'W_' + letter).weight.copy_(torch.tensor(packed).T)
        model.attention.W_O.weight.copy_(torch.tensor(worksheet['headsLesson']['W_O']).T)
        model.hidden.weight.copy_(torch.tensor(worksheet['W_hidden']).T)
        model.hidden.bias.copy_(torch.tensor(worksheet['b_hidden']))
        model.readout.weight.copy_(torch.tensor(worksheet['W_vocab']).T)
        model.readout.bias.copy_(torch.tensor(worksheet['b_vocab']))
    return model


In [4]:
model = TinyMultiHeadLM(vocab_size=20, context=10,
                        width=4, heads=2, hidden=8)
load_worksheet_weights(model, worksheet)

TinyMultiHeadLM(
  (token_embedding): Embedding(20, 4)
  (position_embedding): Embedding(10, 4)
  (attention): ScratchMultiHead(
    (W_Q): Linear(in_features=4, out_features=4, bias=False)
    (W_K): Linear(in_features=4, out_features=4, bias=False)
    (W_V): Linear(in_features=4, out_features=4, bias=False)
    (W_O): Linear(in_features=4, out_features=4, bias=False)
  )
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (readout): Linear(in_features=8, out_features=20, bias=True)
)

<a id="s05-batch"></a>
## Two input windows form one batch

[Matching slide](../../part3.html?present#s05/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 185" role="img" aria-label="Example; Ten token IDs; Observed next token" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Example; Ten token IDs; Observed next token</title><g data-cell-left="20" data-cell-right="190"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Example</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Ten token IDs</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Observed next token</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="190"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 1, 2, 3, 0, 4, 5, 6, 7, 0]</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="190"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">cheque</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[8, 9, 0, 10, 11, 0, 5, 6, 7, 0]</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">teller</text></g><path d="M20 165 H1140" stroke="var(--line)"/></svg>

These are integer IDs, not embeddings. Each example has one observed target.

In [5]:
X = torch.tensor([river_ids, cheque_ids])
y = torch.tensor([word_to_id["water"], word_to_id["teller"]])

<a id="s05-embed-map"></a>
## Find the embedding lookup on the map

[Matching slide](../../part3.html?present#s05/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="397.5" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="192" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₁ · K₁ · V₁</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="312" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₂ · K₂ · V₂</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

We have token IDs. Next, look up their learned rows and add the position rows. Both heads will read the result.

<a id="s05-embed"></a>
## Look up token rows and add position rows

[Matching slide](../../part3.html?present#s05/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token IDs [2,10] → Token + position rows → E [2,10,4]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token IDs [2,10] → Token + position rows → E [2,10,4]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token IDs [2,10]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">integers</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position rows</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">lookup, then add</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [2,10,4]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">floating-point vectors</text></g></svg>

The embedding tables are learned parameters. E contains floating-point vectors with shape [2,10,4].

In [6]:
positions = torch.arange(X.shape[1])
E = model.token_embedding(X) + model.position_embedding(positions)

<a id="s05-project"></a>
## Project the full input three times

[Matching slide](../../part3.html?present#s05/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E [2,10,4] → Three matrices → Q, K, V" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E [2,10,4] → Three matrices → Q, K, V</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [2,10,4]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">same input</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Three matrices</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">each 4 × 4</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q, K, V</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">each [2,10,4]</text></g></svg>

Different learned matrices give matching queries, matching keys and transmitted values.

In [7]:
Q = model.attention.W_Q(E)
K = model.attention.W_K(E)
V = model.attention.W_V(E)

<a id="s05-split"></a>
## Make the head axis explicit

[Matching slide](../../part3.html?present#s05/7/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="[2,10,4] → [2,10,2,2] → [2,2,10,2]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>[2,10,4] → [2,10,2,2] → [2,2,10,2]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,10,4]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, D</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,10,2,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, H, d_head</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,2,10,2]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, H, T, d_head</text></g></svg>

reshape groups projected coordinates; transpose puts heads before tokens. Apply the same operation to K and V.

In [8]:
Q = Q.reshape(2, 10, 2, 2).transpose(1, 2)
K = K.reshape(2, 10, 2, 2).transpose(1, 2)
V = V.reshape(2, 10, 2, 2).transpose(1, 2)

<a id="s05-scores"></a>
## Compute every query–key pair within each head

[Matching slide](../../part3.html?present#s05/8/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Q [2,2,10,2] → Kᵀ [2,2,2,10] → Scores [2,2,10,10]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Q [2,2,10,2] → Kᵀ [2,2,2,10] → Scores [2,2,10,10]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q [2,2,10,2]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">query rows</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-k)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-k)" font-size="24" text-anchor="middle" font-weight="650">Kᵀ [2,2,2,10]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source columns</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Scores [2,2,10,10]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">two 10 × 10 grids</text></g></svg>

The first two axes keep examples and heads separate. Matrix multiplication contracts only the matching-coordinate axis.

In [9]:
scores = Q @ K.transpose(-2, -1)
scores = scores / math.sqrt(2)

<a id="s05-mask"></a>
## Block future sources in every head

[Matching slide](../../part3.html?present#s05/9/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Receiving position; Allowed source positions" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Receiving position; Allowed source positions</title><g data-cell-left="20" data-cell-right="470"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Receiving position</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Allowed source positions</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1, 2</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1, 2, 3</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">10</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1 through 10</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

True marks a blocked entry. The same causal mask broadcasts across both examples and both heads.

In [10]:
future = torch.ones(10, 10, dtype=torch.bool).triu(1)
scores = scores.masked_fill(future, float("-inf"))

<a id="s05-softmax"></a>
## Each row becomes a distribution over sources

[Matching slide](../../part3.html?present#s05/10/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><rect x="120" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><rect x="221" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><rect x="322" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><rect x="423" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><rect x="524" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><rect x="625" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><rect x="726" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><rect x="827" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><rect x="928" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><rect x="1029" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="145" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

The final axis indexes source tokens. Every allowed row sums to one; masked entries have weight zero.

In [11]:
A = scores.softmax(dim=-1)
assert A.shape == (2, 2, 10, 10)

In [12]:
expected = torch.tensor([[h['A'] for h in worksheet['headsLesson']['cases'][name]['heads']]
                         for name in ['river', 'cheque']])
torch.testing.assert_close(A, expected)
torch.testing.assert_close(A.sum(-1), torch.ones(2, 2, 10))
assert not A.triu(1).any()
print('All 400 head weights match the worksheet; no future source receives weight.')

All 400 head weights match the worksheet; no future source receives weight.


<a id="s05-mix"></a>
## Multiply the weights by the values

[Matching slide](../../part3.html?present#s05/11/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="A [2,2,10,10] → V [2,2,10,2] → Messages [2,2,10,2]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>A [2,2,10,10] → V [2,2,10,2] → Messages [2,2,10,2]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">A [2,2,10,10]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source weights</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">V [2,2,10,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source content</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Messages [2,2,10,2]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one row per query/head</text></g></svg>

The source-token axis is summed out. Each head keeps its own two-coordinate message.

In [13]:
messages = A @ V

<a id="s05-join"></a>
## Put each token’s head messages side by side

[Matching slide](../../part3.html?present#s05/12/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="[2,2,10,2] → [2,10,2,2] → [2,10,4]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>[2,2,10,2] → [2,10,2,2] → [2,10,4]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,2,10,2]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, H, T, d_head</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,10,2,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, H, d_head</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,10,4]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, D</text></g></svg>

Transpose before reshaping. A direct reshape of the original layout would mix token rows with head rows.

In [14]:
joined = messages.transpose(1, 2).contiguous()
joined = joined.reshape(2, 10, 4)

<a id="s05-output-map"></a>
## Return to the shared prediction path

[Matching slide](../../part3.html?present#s05/13/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="192" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₁ · K₁ · V₁</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="312" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₂ · K₂ · V₂</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

The two head messages are joined. W_O mixes them, and the residual adds that update to the original E.

<a id="s05-output"></a>
## Project, add, and keep the final row

[Matching slide](../../part3.html?present#s05/14/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Joined messages → W_O + residual → Last row → MLP" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Joined messages → W_O + residual → Last row → MLP</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Joined messages</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2,10,4]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">updated E′ [2,10,4]</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">logits [2,20]</text></g></svg>

W_O combines the heads. The residual keeps E. Only the last updated row feeds this next-token loss.

In [15]:
delta = model.attention.W_O(joined)
updated = E + delta
logits = model.readout(F.relu(model.hidden(updated[:, -1])))

In [16]:
expected = torch.tensor([worksheet['headsLesson']['cases'][name]['logits'][-1]
                         for name in ['river', 'cheque']])
torch.testing.assert_close(logits, expected)
torch.testing.assert_close(model(X), logits)
print('All 40 vocabulary logits match the independent worksheet.')

All 40 vocabulary logits match the independent worksheet.


<a id="s06-break"></a>
## The rest of the learning loop is familiar

[Matching slide](../../part3.html?present#s06/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

More heads change the context update, not the definition of the next-token target.

<a id="s06-loss"></a>
## Score the two observed targets

[Matching slide](../../part3.html?present#s06/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Input X → Two heads + MLP → Logits [2,20] → Loss" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input X → Two heads + MLP → Logits [2,20] → Loss</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Input X</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2 × 10 token IDs</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Two heads + MLP</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one prediction/example</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Logits [2,20]</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">targets: water, teller</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Loss</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one scalar</text></g></svg>

Cross-entropy reads logits and observed token IDs. Do not sample a generated token to make the training target.

In [17]:
logits = model(X)
loss = F.cross_entropy(logits, y)

<a id="s06-update"></a>
## One loss trains all the projections

[Matching slide](../../part3.html?present#s06/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Loss → Gradients → Optimizer → Updated weights" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Loss → Gradients → Optimizer → Updated weights</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Loss</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">from the same targets</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Gradients</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">all learned parameters</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Optimizer</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one update</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Updated weights</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">used by the next batch</text></g></svg>

Heads are not assigned jobs or separate labels. They receive gradients from the same prediction loss.

In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [19]:
assert all(p.grad is not None and p.grad.isfinite().all() for p in model.parameters())
print('All learned tables, head projections and prediction layers received finite gradients.')

All learned tables, head projections and prediction layers received finite gradients.


<a id="s06-prompt"></a>
## Start generation from known token IDs

[Matching slide](../../part3.html?present#s06/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="River prefix → Vocabulary lookup → History [1,10]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>River prefix → Vocabulary lookup → History [1,10]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">River prefix</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">the ten known words</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Vocabulary lookup</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">same vocabulary</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">History [1,10]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">no observed next token</text></g></svg>

We already encoded the river sentence. A real application tokenizes and looks up a new prompt with the same vocabulary.

In [20]:
history = torch.tensor([river_ids])
model.eval()

TinyMultiHeadLM(
  (token_embedding): Embedding(20, 4)
  (position_embedding): Embedding(10, 4)
  (attention): ScratchMultiHead(
    (W_Q): Linear(in_features=4, out_features=4, bias=False)
    (W_K): Linear(in_features=4, out_features=4, bias=False)
    (W_V): Linear(in_features=4, out_features=4, bias=False)
    (W_O): Linear(in_features=4, out_features=4, bias=False)
  )
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (readout): Linear(in_features=8, out_features=20, bias=True)
)

<a id="s06-infer"></a>
## Generate one token, then repeat

[Matching slide](../../part3.html?present#s06/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known IDs → Crop to context → Model logits → Choose + append" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known IDs → Crop to context → Model logits → Choose + append</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known IDs</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">no future target</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Crop to context</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">latest 10 in this toy</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Model logits</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">final row only</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Choose + append</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">new known prefix</text></g></svg>

At inference, weights stay fixed. The current prefix becomes longer; we keep only the configured context window.

In [21]:
with torch.no_grad():
    logits = model(history[:, -model.context:])
next_id = logits.argmax(dim=-1, keepdim=True)
history = torch.cat([history, next_id], dim=1)

<a id="s06-map"></a>
## Trace one request through the full model

[Matching slide](../../part3.html?present#s06/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="120.0" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="397.5" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₁ · K₁ · V₁</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₂ · K₂ · V₂</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="690.0" y="250" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

Start at the known IDs. Name the shape at each arrow. Where does the observed target enter during training? It enters only at the loss.

<a id="s07-break"></a>
## The same operation, packaged by PyTorch

[Matching slide](../../part3.html?present#s07/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Which boxes does nn.MultiheadAttention replace?

<a id="s07-api-map"></a>
## Replace the head calculation, not the whole model

[Matching slide](../../part3.html?present#s07/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₁ · K₁ · V₁</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q₂ · K₂ · V₂</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

The PyTorch layer replaces both heads, their concatenation and W_O. We still add the residual and use our prediction MLP.

<a id="s07-api"></a>
## Create the multi-head attention layer

[Matching slide](../../part3.html?present#s07/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Input E → nn.MultiheadAttention → Projected update ΔE" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input E → nn.MultiheadAttention → Projected update ΔE</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Input E</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[B,T,4]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">nn.MultiheadAttention</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2 heads × 2 coordinates</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Projected update ΔE</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[B,T,4]</text></g></svg>

batch_first=True means [B,T,D]. bias=False and dropout=0 match our scratch implementation.

In [22]:
mha = nn.MultiheadAttention(
    embed_dim=4, num_heads=2, bias=False,
    dropout=0.0, batch_first=True)

<a id="s07-call"></a>
## Pass E as query, key and value input

[Matching slide](../../part3.html?present#s07/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E, E, E → nn.MultiheadAttention → ΔE and A" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E, E, E → nn.MultiheadAttention → ΔE and A</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E, E, E</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">unprojected input rows</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">nn.MultiheadAttention</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q/K/V, softmax, mix, W_O</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">ΔE and A</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">not a residual yet</text></g></svg>

PyTorch performs the learned projections internally. average_attn_weights=False preserves the head axis in the returned weight tensor.

In [23]:
delta, A = mha(E, E, E, attn_mask=future,
               average_attn_weights=False)
updated = E + delta

<a id="s07-check"></a>
## Compare the same weights, not two random layers

[Matching slide](../../part3.html?present#s07/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Quantity; Expected shape" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Quantity; Expected shape</title><g data-cell-left="20" data-cell-right="580"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Quantity</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Expected shape</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Projected update ΔE</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,10,4]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Separate head weights A</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,2,10,10]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Residual E + ΔE</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,10,4]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The notebook copies the scratch Q/K/V and W_O parameters into PyTorch, then checks both outputs numerically.

In [24]:
mha = copy_to_pytorch(model.attention)
delta_scratch, A_scratch = model.attention(E)
delta_api, A_api = mha(E, E, E, attn_mask=future,
                      average_attn_weights=False)
torch.testing.assert_close(delta_api, delta_scratch)

In [25]:
torch.testing.assert_close(A_api, A_scratch)
print('Scratch and PyTorch updates agree:', tuple(delta_api.shape))
print('Individual head weights agree:', tuple(A_api.shape))

Scratch and PyTorch updates agree: (2, 10, 4)
Individual head weights agree: (2, 2, 10, 10)


<a id="s07-boundary"></a>
## What the attention block does not include

[Matching slide](../../part3.html?present#s07/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token + position → Multi-head attention → Residual → Prediction MLP" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token + position → Multi-head attention → Residual → Prediction MLP</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">outside the API</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Multi-head attention</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">returns projected ΔE</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Residual</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">add E ourselves</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Prediction MLP</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">outside the API</text></g></svg>

nn.MultiheadAttention does not add position embeddings, the residual, a vocabulary head or a training loop.

<a id="s08-break"></a>
## Do more heads help this experiment?

[Matching slide](../../part3.html?present#s08/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Move from the small worksheet to the actual TinyStories checkpoints.

<a id="s08-width"></a>
## More heads need not mean more parameters

[Matching slide](../../part3.html?present#s08/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 185" role="img" aria-label="Setting; Total width; Heads; Width per head" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Setting; Total width; Heads; Width per head</title><g data-cell-left="20" data-cell-right="440"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Setting</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Total width</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Heads</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Width per head</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">16</text></g><path d="M20 165 H1140" stroke="var(--line)"/></svg>

Q, K, V and W_O remain 64×64. The number of attention patterns grows, while each pattern uses fewer matching coordinates.

<a id="s08-scores"></a>
## Four heads improved held-out prediction here

[Matching slide](../../part3.html?present#s08/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Model; Test cross-entropy ↓; Test perplexity ↓" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Model; Test cross-entropy ↓; Test perplexity ↓</title><g data-cell-left="20" data-cell-right="430"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Model</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Test cross-entropy ↓</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Test perplexity ↓</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.445</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">31.34</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">MLP</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.943</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">51.59</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.360</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">28.78</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Three-seed means. Same stories, tokenizer, context and update budget; validation-selected checkpoints. Perplexity = exp(cross-entropy). Lower is better. This does not guarantee better text for every prompt.

<a id="s08-costs"></a>
## Compare the cost as well as the score

[Matching slide](../../part3.html?present#s08/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Model; Parameters; Training per seed" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Model; Parameters; Training per seed</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Model</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Parameters</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Training per seed</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">MLP</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2,332,832</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">58.2 s</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1,321,120</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">65.6 s</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1,321,120</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">69.7 s</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Mean training time on Apple M2 Max/MPS, including validation checks. Both attention models have equal parameter counts; the MLP is larger.

<a id="s08-demo"></a>
## Try the same prompt with all three models

[Matching slide](../../part3.html?present#s08/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Training prefix → Held-out prefix → Outside-domain prompt" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Training prefix → Held-out prefix → Outside-domain prompt</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Training prefix</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">familiar text</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Held-out prefix</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">new story, same task</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Outside-domain prompt</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">different kind of text</text></g></svg>

<a href="../../word-lab/" target="_blank" rel="noopener">Open the live browser demo ↗</a> Compare continuations, vocabulary coverage and generation time. This runs real checkpoints with WebGPU or WASM.

<a id="s08-notebooks"></a>
## Run the calculation yourself

[Matching slide](../../part3.html?present#s08/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Resource; What to do" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Resource; What to do</title><g data-cell-left="20" data-cell-right="440"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Resource</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">What to do</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Notebook 7</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Follow every operation and check PyTorch parity</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Notebook 6</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Inspect the trained four-head checkpoint and results</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Part 2B · optional</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Gradients, normalization, full blocks and context cost</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

<a href="../../notebooks/wordlm/07_multihead_step_by_step.html">Step-by-step notebook</a> · <a href="../../notebooks/wordlm/06_multihead_comparison.html">Measured experiment</a> · <a href="../../part2b.html">Optional reference</a>

<a id="s08-next"></a>
## Next: read a different sequence

[Matching slide](../../part3.html?present#s08/7/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="French decoder row → Query → English keys + values" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>French decoder row → Query → English keys + values</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">French decoder row</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">the next output token</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Query</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">what do I need?</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">English keys + values</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">where should I read?</text></g></svg>

So far, Q, K and V came from the same sequence. <a href="../../part4.html">Part IV changes that: cross-attention reads another sequence.</a>

## Continue with the trained experiment

[Notebook 6](06_multihead_comparison.html) inspects the four-head TinyStories
checkpoint, per-head weights and all three-seed results. The browser demo loads
the actual exported checkpoints and measures generation on your device.

References: [Attention Is All You Need, §3.2.2](https://arxiv.org/abs/1706.03762)
and [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html).
The diagrams and worked numbers here are original adaptations of our Part II example.